# Pet Segmentation — a mask for every pixel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s7-pet-segmentation.ipynb)

**MS2A — Machine Learning Practice · Session 7, Computer Vision 2**

Classification gives one label per image; **segmentation** gives one per *pixel*. Here: which
pixels of a photo are the cat or the dog, and which are background —
[challenge 197](https://ml-arena.com/viewchallenge/197) on ML-Arena, scored by **mean IoU**.

| Section | What you do |
|---|---|
| 0–2 | Download the data, look at the images and the masks |
| 3 | Hold out a validation set, write down the metric |
| 4 | **A small U-Net from scratch** — first submission |
| 5 | **Fine-tune a pretrained U-Net** (ImageNet ResNet-18 encoder) — second submission |

> In Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom
> (about 10 minutes). Edit your API token (your **Profile** page on ml-arena.com), then
> write the cells marked **TODO**: the core of the small U-Net in section 4 (forward pass,
> loss, training step, prediction). Each TODO is followed by a cell that checks it, so an
> error shows up where it was made.

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk>=4.4" segmentation-models-pytorch

import mlarena
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random, sys, time, zipfile
import torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
CHALLENGE_ID = 197

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. Get the data

`train.zip` holds the training photos (`images/<id>.jpg`) and their masks (`masks/<id>.png`),
`test_images.zip` the 1,500 photos you predict. `seg_metric.py` is the leaderboard's own
code: the submission format and the metric. We import it to score locally.

In [ ]:
client.download_dataset(CHALLENGE_ID, "data/")
zipfile.ZipFile("data/train.zip").extractall("data/train")
zipfile.ZipFile("data/test_images.zip").extractall("data/test")
sys.path.insert(0, "data")
import seg_metric as sm

train_ids = sorted(p.stem for p in Path("data/train/images").glob("*.jpg"))
test_ids = sorted(p.stem for p in Path("data/test/images").glob("*.jpg"))
X_all = np.stack([np.array(Image.open(f"data/train/images/{i}.jpg")) for i in train_ids])
M_all = np.stack([np.array(Image.open(f"data/train/masks/{i}.png")) for i in train_ids])
X_test = np.stack([np.array(Image.open(f"data/test/images/{i}.jpg")) for i in test_ids])
print("images", X_all.shape, X_all.dtype, "| masks", M_all.shape, "values", np.unique(M_all))
print("test images", X_test.shape)

## 2. Look at it

A mask has one value per pixel: **0** background, **1** pet, **2** a thin **border** band
around the pet, where the annotators could not decide. The border is **not scored**, and we
will not train on it either.

In [ ]:
colors = np.array([[0, 0, 0], [255, 140, 0], [255, 255, 255]], dtype=np.uint8)  # bg, pet, border

def show(images, masks_list, titles, n=6):
    """One column per image: the photo, then one row per mask set (labels 0/1/2 or bool)."""
    rows = 1 + len(masks_list)
    fig, axes = plt.subplots(rows, n, figsize=(2.2 * n, 2.2 * rows))
    for j in range(n):
        axes[0, j].imshow(images[j])
        for r, masks in enumerate(masks_list, start=1):
            axes[r, j].imshow(colors[masks[j].astype(np.uint8)])
        for r in range(rows):
            axes[r, j].axis("off")
    for r, t in enumerate(["photo"] + titles):
        axes[r, 0].set_title(t, fontsize=9, loc="left")
    plt.tight_layout(); plt.show()

pick = random.Random(0).sample(range(len(X_all)), 6)
show(X_all[pick], [M_all[pick]], ["mask: black background, orange pet, white border"])

How much of an image is pet? This decides what a trivial answer scores.

In [ ]:
scored = M_all != 2
pet_share = ((M_all == 1) & scored).sum((1, 2)) / scored.sum((1, 2))
plt.hist(pet_share, bins=50); plt.xlabel("share of pet pixels per image"); plt.show()
print(f"pet = {(M_all == 1).sum() / scored.sum():.1%} of the scored pixels")
print("predict all background:", sm.score_masks(M_all, np.zeros(M_all.shape, bool)))

## 3. Validation set and metric

The test masks are private, so we keep 15 % of the training images aside. For each class *c*
(background, pet), over all scored pixels of all images together:

$$\text{IoU}(c) = \frac{|\text{pred}=c \;\cap\; \text{true}=c|}{|\text{pred}=c \;\cup\; \text{true}=c|}\qquad \text{score} = \frac{\text{IoU(background)} + \text{IoU(pet)}}{2}$$

Predicting *all background* already gets IoU(background) ≈ 0.66, so the floor is about 0.33,
not 0. `sm.score_masks(true, pred)` computes it exactly as the leaderboard does.

In [ ]:
idx = np.random.RandomState(0).permutation(len(X_all))
n_val = len(idx) * 15 // 100
X_val, M_val = X_all[idx[:n_val]], M_all[idx[:n_val]]
X_tr, M_tr = X_all[idx[n_val:]], M_all[idx[n_val:]]
print(len(X_tr), "train,", len(X_val), "validation")

## 4. A small U-Net from scratch

The **encoder** halves the resolution four times (128 → 8) while widening the channels:
it sees more and more context, at a coarser and coarser grid. The **decoder** goes back up
with transposed convolutions, and at each scale **concatenates** the encoder's map of the same
size — the *skip connections* that give back the fine detail the bottleneck lost.
The last 1×1 convolution outputs 2 scores per pixel (background, pet).

In [ ]:
def block(c_in, c_out):
    """Two 3x3 convolutions, each followed by BatchNorm and ReLU: the U-Net's unit."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

class SmallUNet(nn.Module):
    def __init__(self, n_classes=2, c=16):
        super().__init__()
        self.enc1, self.enc2 = block(3, c), block(c, 2 * c)               # 128, 64
        self.enc3, self.enc4 = block(2 * c, 4 * c), block(4 * c, 8 * c)   # 32, 16
        self.bottom = block(8 * c, 16 * c)                                # 8 x 8
        self.up4, self.dec4 = nn.ConvTranspose2d(16 * c, 8 * c, 2, stride=2), block(16 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), block(2 * c, c)
        self.head = nn.Conv2d(c, n_classes, 1)                            # one score per class and pixel

    def forward(self, x):
        # TODO
        # encoder: e1 = enc1(x), then e2, e3, e4 and the bottom b, each after a 2x2 max pool
        #          (F.max_pool2d): 128 -> 64 -> 32 -> 16 -> 8;
        # decoder: d4 = dec4 of [up4(b), e4] concatenated on the channel axis (torch.cat(..., 1)),
        #          then d3, d2, d1 the same way down to e1;
        # return head(d1): (batch, 2, 128, 128) logits.
        ...


In [ ]:
model = SmallUNet(c=16)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters")
out = model(torch.zeros(2, 3, 128, 128))
assert out is not None and out.shape == (2, 2, 128, 128), getattr(out, "shape", out)
print("SmallUNet: OK")


The loss is a **cross-entropy per pixel**: each pixel is a 2-class classification
(background, pet), averaged over the pixels. Border pixels (label 2) must count for nothing.

In [ ]:
# TODO: cross-entropy over the 2 classes of every pixel, with label 2 (border) ignored.
# nn.CrossEntropyLoss takes logits (N, 2, H, W) and labels (N, H, W): see its ignore_index.
loss_fn = ...


In [ ]:
logits = torch.zeros(1, 2, 1, 3, requires_grad=True)       # 3 pixels, both scores 0
labels = torch.tensor([[[1, 0, 2]]])                          # pet, background, border
loss = loss_fn(logits, labels)
assert abs(loss.item() - np.log(2)) < 1e-5, f"loss {loss.item():.4f}, expected log 2 = 0.6931 (2 pixels scored)"
loss.backward()
assert logits.grad[..., 2].abs().sum() == 0, "the border pixel must give no gradient"
assert logits.grad[..., :2].abs().sum() > 0
print("loss_fn: OK")


The loop: Adam with a one-cycle learning rate, the **cross-entropy per pixel** below, random horizontal flips as the only
augmentation, and the weights of the best validation epoch kept.

In [ ]:
def to_input(images, mean=None, std=None):
    """uint8 (N, H, W, 3) -> float tensor (N, 3, H, W) in [0, 1], optionally normalised."""
    x = torch.as_tensor(images).permute(0, 3, 1, 2).float().contiguous() / 255
    if mean is not None:
        x = (x - torch.tensor(mean).view(1, 3, 1, 1)) / torch.tensor(std).view(1, 3, 1, 1)
    return x

def train(model, epochs, lr, mean=None, std=None, batch_size=32):
    """Adam + cross-entropy, border pixels ignored, random horizontal flips.
    Keeps the weights of the epoch with the best validation mIoU."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * (len(X_tr) // batch_size))
    X, Y = to_input(X_tr, mean, std), torch.as_tensor(M_tr).long()
    gen = torch.Generator().manual_seed(0)
    history, best = [], (-1.0, None)
    for epoch in range(epochs):
        model.train()
        t0, losses = time.time(), []
        perm = torch.randperm(len(X), generator=gen)
        for i in range(0, len(X) - batch_size + 1, batch_size):
            idx = perm[i:i + batch_size]
            x, y = X[idx], Y[idx]
            flip = torch.rand(len(idx), generator=gen) < 0.5   # augmentation: mirror half the batch
            x[flip], y[flip] = x[flip].flip(-1), y[flip].flip(-1)
            # TODO: the loss of the model's logits for x against y (both moved to `device`),
            # then reset the gradients, backpropagate, step the optimiser and the scheduler.
            loss = ...
            losses.append(loss.item())
        val = sm.score_masks(M_val, predict(model, X_val, mean, std))
        history.append({"epoch": epoch + 1, "loss": np.mean(losses), **val})
        print(f"epoch {epoch + 1:2d}  loss {np.mean(losses):.3f}  val mIoU {val['miou']:.4f}"
              f"  (pet IoU {val['iou_pet']:.3f})  {time.time() - t0:.0f}s")
        if val["miou"] > best[0]:
            best = (val["miou"], {k: v.detach().clone() for k, v in model.state_dict().items()})
    model.load_state_dict(best[1])
    return pd.DataFrame(history)

@torch.no_grad()
def predict(model, images, mean=None, std=None, batch_size=64):
    """uint8 images (N, H, W, 3) -> bool pet masks (N, H, W)."""
    model.eval()
    out = []
    for i in range(0, len(images), batch_size):
        logits = model(to_input(images[i:i + batch_size], mean, std).to(device))
        # TODO: logits (batch, 2, H, W) -> bool pet masks (batch, H, W), as a numpy array.
        out.append(...)
    return np.concatenate(out)

In [ ]:
class LeftHalfPet(nn.Module):
    """A fake model: the pet score wins on the left half of every image."""
    def forward(self, x):
        logits = torch.zeros(len(x), 2, *x.shape[2:], device=x.device)
        logits[:, 1, :, : x.shape[3] // 2] = 1
        return logits

p = predict(LeftHalfPet(), X_val[:3])
half = X_val.shape[2] // 2
assert isinstance(p, np.ndarray) and p.dtype == bool and p.shape == X_val[:3].shape[:3], (type(p), getattr(p, "shape", None))
assert p[:, :, :half].all() and not p[:, :, half:].any(), "pet must be predicted where its score is the larger one"
print("predict: OK")


In [ ]:
history = train(model, epochs=15, lr=0.003)
history.plot(x="epoch", y=["loss", "miou"], secondary_y="miou", figsize=(7, 3)); plt.show()
assert history["miou"].max() > 0.5, "no better than predicting all background (about 0.33): check the training step"


Validation predictions next to the truth. Where does it go wrong?

In [ ]:
pred_val = predict(model, X_val)
print(sm.score_masks(M_val, pred_val))
show(X_val[:6], [M_val[:6], pred_val[:6]], ["truth", "small U-Net"])

### 4.1 First submission

One row per test image: `image_id,mask_rle`, the pet pixels run-length encoded
(`sm.rle_encode`). The leaderboard decodes it with `sm.rle_decode`.

In [ ]:
def submit(pred, name):
    """pred: bool masks (1500, 128, 128) in test_ids order. Writes submission.csv and submits it."""
    pd.DataFrame({"image_id": test_ids, "mask_rle": [sm.rle_encode(m) for m in pred]})\
      .to_csv("submission.csv", index=False)
    assert len(sm.read_rle_csv("submission.csv", image_ids=test_ids)) == len(test_ids)  # the format check
    print(client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"], submission_name=name))

submit(predict(model, X_test), "small-unet-scratch")

## 5. Fine-tune a pretrained U-Net

Same U-Net shape, but the encoder is a **ResNet-18 trained on ImageNet**: it already
knows edges, fur and eyes. Only the decoder starts from random weights. Two things change:
the inputs are normalised with ImageNet's mean and std (what the encoder saw in training),
and the learning rate is lower, so the pretrained features are adjusted rather than erased.

In [ ]:
import segmentation_models_pytorch as smp

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
pretrained = smp.Unet("resnet18", encoder_weights="imagenet", classes=2)
print(f"{sum(p.numel() for p in pretrained.parameters()) / 1e6:.1f} M parameters")
history_ft = train(pretrained, epochs=8, lr=0.001, mean=IMAGENET_MEAN, std=IMAGENET_STD)

In [ ]:
pred_val_ft = predict(pretrained, X_val, IMAGENET_MEAN, IMAGENET_STD)
print("scratch   ", sm.score_masks(M_val, pred_val))
print("fine-tuned", sm.score_masks(M_val, pred_val_ft))
show(X_val[:6], [M_val[:6], pred_val[:6], pred_val_ft[:6]], ["truth", "small U-Net", "fine-tuned"])

**Questions**
- The pretrained encoder never saw a segmentation mask. Why does it help this much, and with
  how many epochs?
- Which images does the fine-tuned model still get wrong? Small pets, pets cut by the frame,
  cluttered backgrounds?

Submit it:

In [ ]:
submit(predict(pretrained, X_test, IMAGENET_MEAN, IMAGENET_STD), "unet-resnet18-finetuned")

## Going further

Change one thing, compare the validation mIoU, submit only if it improved:
- a bigger encoder: `smp.Unet("resnet34", ...)`, or another architecture: `smp.DeepLabV3Plus`;
- more epochs, or retraining on all the training images once the settings are chosen;
- **test-time augmentation**: average the logits of the image and its mirror;
- upsample the inputs to 256 × 256 for the pretrained encoder, and the logits back to 128;
- a Dice loss added to the cross-entropy.